# Step 2 of RAG: Storing Chunks in ChromaDB

**RAG** (Retrieval-Augmented Generation) pipeline recap:

1. Document Loading & Chunking (Step 1)
2. **Storing chunks in a Vector Database** ← *this notebook*
3. Retrieval (searching the vector database for relevant chunks)
4. Passing retrieved chunks to an LLM to generate the final answer

This notebook is self-contained — it recreates a sample document, chunks it (recap of Step 1),
and then focuses fully on **storing those chunks in ChromaDB** the right way: with embeddings,
useful metadata, and stable IDs so we can retrieve, update, or trace every chunk back to its source.

### What you will learn
1. Recap: create and chunk a document
2. Why metadata (source, chunk index, etc.) matters when storing chunks
3. Setting up a ChromaDB collection with a Hugging Face embedding function
4. Storing chunks with meaningful IDs and metadata
5. Verifying what got stored
6. Querying stored chunks (a preview of Step 3: Retrieval)
7. Adding a second document to show multi-document storage

### Setup — Install Required Libraries

In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

## 1. Recap — Create and Chunk a Document (Step 1)

We regenerate a small sample document and split it into chunks using
`RecursiveCharacterTextSplitter`, exactly as in the Step 1 notebook.

In [2]:
import os

os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

with open("docs/ai_overview.txt", "w") as f:
    f.write(article_text)

print("Document created: docs/ai_overview.txt")

Document created: docs/ai_overview.txt


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)

print("Number of chunks created:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)

C:\Users\Lahari\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of chunks created: 9

--- Chunk 0 (36 chars) ---
Artificial Intelligence: An Overview

--- Chunk 1 (270 chars) ---
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

--- Chunk 2 (291 chars) ---
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters

--- Chunk 3 (26 chars) ---
to recommendation systems.

--- Chunk 4 (260 chars) ---
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing f

## 2. Why Metadata Matters When Storing Chunks

When we store a chunk in a vector database, we should **never** store just the raw text. We should
also attach metadata so that later, when a chunk is retrieved, we know:

| Metadata field | Why it matters |
|---|---|
| `source` | Which original file/document this chunk came from |
| `chunk_index` | The chunk's position/order within that document |
| `char_count` | Useful for debugging or filtering overly short/long chunks |

This makes results traceable — a RAG answer can cite "this came from `ai_overview.txt`, chunk 3".

In [4]:
chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(chunks)
]

chunk_ids = [f"ai_overview_chunk_{i}" for i in range(len(chunks))]

print("Example metadata for chunk 0:", chunk_metadatas[0])
print("Example id for chunk 0:", chunk_ids[0])

Example metadata for chunk 0: {'source': 'ai_overview.txt', 'chunk_index': 0, 'char_count': 36}
Example id for chunk 0: ai_overview_chunk_0


## 3. Set Up ChromaDB with a Hugging Face Embedding Function

We use the free `all-MiniLM-L6-v2` model so ChromaDB automatically converts every chunk's text into
an embedding vector as we add it.

In [5]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("Collection created:", collection.name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1814.56it/s]


Collection created: rag_chunks


## 4. Store the Chunks in ChromaDB

We pass `documents` (the chunk text), `metadatas`, and `ids` all together. ChromaDB embeds each
chunk automatically using the embedding function we attached above.

In [6]:
collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("Total chunks stored in ChromaDB:", collection.count())

Total chunks stored in ChromaDB: 9


## 5. Verify What Got Stored

Always double-check after a storage step — inspect a few chunks along with their metadata.

In [7]:
stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

for cid, doc, meta in zip(stored["ids"], stored["documents"], stored["metadatas"]):
    print(f"ID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc[:100]}...")
    print("-" * 60)

ID: ai_overview_chunk_0
Metadata: {'source': 'ai_overview.txt', 'chunk_index': 0, 'char_count': 36}
Text: Artificial Intelligence: An Overview...
------------------------------------------------------------
ID: ai_overview_chunk_1
Metadata: {'char_count': 270, 'chunk_index': 1, 'source': 'ai_overview.txt'}
Text: Artificial Intelligence, or AI, is the field of computer science focused on building systems
that ca...
------------------------------------------------------------
ID: ai_overview_chunk_2
Metadata: {'chunk_index': 2, 'char_count': 291, 'source': 'ai_overview.txt'}
Text: Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardc...
------------------------------------------------------------
ID: ai_overview_chunk_3
Metadata: {'chunk_index': 3, 'source': 'ai_overview.txt', 'char_count': 26}
Text: to recommendation systems....
------------------------------------------------------------
ID: ai_overview_chunk_4
Metadata: {'source': 'ai_ove

## 6. Query the Stored Chunks (Preview of Step 3: Retrieval)

Now that our chunks live in ChromaDB, we can search them by meaning. This is the foundation of the
next RAG step — retrieving the most relevant chunks for a user's question.

In [8]:
results = collection.query(
    query_texts=["How does RAG combine search with language models?"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.2446 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
------------------------------------------------------------
Distance: 0.6552 | Source: ai_overview.txt | Chunk #8
Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
------------------------------------------------------------


**Discussion Point:** The `distance` score tells us how relevant each chunk is (lower = more
relevant). In a full RAG system, these top chunks would now be inserted into a prompt and sent to an
LLM to generate a grounded final answer — that's Step 3 and Step 4.

## 7. Adding a Second Document (Multi-Document Storage)

Real RAG systems store chunks from many documents in the same collection. The `source` metadata field
is what lets us tell them apart later.

In [9]:
second_doc_text = """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.
"""

with open("docs/embeddings_in_practice.txt", "w") as f:
    f.write(second_doc_text)

second_doc_chunks = splitter.split_text(second_doc_text)

second_doc_metadatas = [
    {
        "source": "embeddings_in_practice.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(second_doc_chunks)
]

second_doc_ids = [f"embeddings_in_practice_chunk_{i}" for i in range(len(second_doc_chunks))]

collection.add(
    documents=second_doc_chunks,
    metadatas=second_doc_metadatas,
    ids=second_doc_ids
)

print("Chunks from second document added:", len(second_doc_chunks))
print("Total chunks now stored in ChromaDB:", collection.count())

Chunks from second document added: 3
Total chunks now stored in ChromaDB: 12


In [10]:
# Query across BOTH documents at once
results = collection.query(
    query_texts=["how do search engines use embeddings"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.1694 | Source: embeddings_in_practice.txt | Chunk #1
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------
Distance: 0.2940 | Source: embeddings_in_practice.txt | Chunk #0
Embeddings and Vector Search in Practice
------------------------------------------------------------
Distance: 0.4229 | Source: ai_overview.txt | Chunk #8
Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
------------------------------------------------------------


**Discussion Point:** Notice the top result correctly comes from `embeddings_in_practice.txt` even
though our collection also contains chunks from `ai_overview.txt`. This is exactly how a RAG system
searches across an entire knowledge base to find the most relevant pieces, regardless of source.

## 8. Filtering Retrieval to a Single Source (Optional but Useful)

Sometimes you only want to search within one document — combine semantic search with a metadata
filter using `where`.

In [11]:
results = collection.query(
    query_texts=["what is deep learning"],
    n_results=2,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.3109 | Source: ai_overview.txt | Chunk #4
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.
------------------------------------------------------------
Distance: 0.5968 | Source: ai_overview.txt | Chunk #1
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.
------------------------------------------------------------


## Key Takeaways (Recap for Students)

| Concept | What We Did |
|---|---|
| Chunk storage | Stored each chunk as a `document` in ChromaDB, not the whole file at once |
| Embedding function | Attached `all-MiniLM-L6-v2` so ChromaDB embeds text automatically on `add()` |
| Metadata | Stored `source`, `chunk_index`, `char_count` with every chunk for traceability |
| IDs | Used predictable ids like `ai_overview_chunk_0` so chunks can be updated/deleted later |
| Multi-document storage | Added a second document into the same collection |
| Filtered retrieval | Used `where={"source": ...}` to search within a single document only |

This collection of stored chunks is now ready for **Step 3: Retrieval**, where we take a user's
question, search this same collection, and pass the top chunks to an LLM to generate the final answer.

### Practice Exercises
1. Add a third document of your own and confirm `collection.count()` reflects all chunks from all three.
2. Re-chunk `ai_overview.txt` with `chunk_size=150` and re-store it under new ids (e.g. `..._v2_chunk_0`)
   — compare retrieval results between the two chunk sizes.
3. Write a helper function `add_document(collection, splitter, text, source_name)` that handles chunking,
   metadata creation, and `collection.add()` in one reusable call.

In [12]:
pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [13]:
import chromadb

# Create / connect to ChromaDB
client = chromadb.PersistentClient(path="./chroma_db")

# Create a new collection
collection = client.get_or_create_collection(
    name="practice1_collection"
)

# -----------------------------
# First document
# -----------------------------

text1 = """
Artificial Intelligence is a technology that allows machines
to perform tasks that normally require human intelligence.
"""

chunks1 = [
    text1[i:i + 100]
    for i in range(0, len(text1), 100)
]

collection.add(
    documents=chunks1,
    ids=[
        f"ai_overview_chunk_{i}"
        for i in range(len(chunks1))
    ],
    metadatas=[
        {
            "source": "ai_overview.txt",
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks1)
    ]
)

# -----------------------------
# Second document
# -----------------------------

text2 = """
Machine Learning is a part of Artificial Intelligence.
It allows computers to learn patterns from data.
"""

chunks2 = [
    text2[i:i + 100]
    for i in range(0, len(text2), 100)
]

collection.add(
    documents=chunks2,
    ids=[
        f"machine_learning_chunk_{i}"
        for i in range(len(chunks2))
    ],
    metadatas=[
        {
            "source": "machine_learning.txt",
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks2)
    ]
)

# -----------------------------
# Third document
# -----------------------------

text3 = """
Python is a popular programming language.
It is widely used in Artificial Intelligence,
Machine Learning, Data Science and Web Development.
"""

chunks3 = [
    text3[i:i + 100]
    for i in range(0, len(text3), 100)
]

collection.add(
    documents=chunks3,
    ids=[
        f"python_chunk_{i}"
        for i in range(len(chunks3))
    ],
    metadatas=[
        {
            "source": "python.txt",
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks3)
    ]
)

# -----------------------------
# Display results
# -----------------------------

print("Third document added successfully!")

print("First document chunks:", len(chunks1))
print("Second document chunks:", len(chunks2))
print("Third document chunks:", len(chunks3))

print("Total chunks in collection:", collection.count())

Third document added successfully!
First document chunks: 2
Second document chunks: 2
Third document chunks: 2
Total chunks in collection: 6


In [15]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="practice2"
)

# Original document text
text = """
Artificial Intelligence is a branch of computer science.
It focuses on creating machines that can perform tasks
that normally require human intelligence.
AI is used in healthcare, education, banking, transportation
and entertainment.
Machine Learning is an important part of Artificial Intelligence.
Deep Learning uses neural networks to learn complex patterns
from large amounts of data.
"""

# Chunk size = 150
chunk_size = 150

chunks = [
    text[i:i+chunk_size]
    for i in range(0, len(text), chunk_size)
]

# Store chunks with new IDs
collection.add(
    documents=chunks,
    ids=[
        f"ai_overview_v2_chunk_{i}"
        for i in range(len(chunks))
    ],
    metadatas=[
        {
            "source": "ai_overview_v2.txt",
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks)
    ]
)

print("Document stored successfully!")
print("Chunk size:", chunk_size)
print("Number of chunks:", len(chunks))
print("Total collection count:", collection.count())

# Retrieval
question = "What is Artificial Intelligence?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

print("\nQuestion:", question)

print("\nRetrieved chunks:")

for i, document in enumerate(results["documents"][0]):
    print("\nChunk", i + 1)
    print(document)

print("\nMetadata:")

for metadata in results["metadatas"][0]:
    print(metadata)

Document stored successfully!
Chunk size: 150
Number of chunks: 3
Total collection count: 3

Question: What is Artificial Intelligence?

Retrieved chunks:

Chunk 1

Artificial Intelligence is a branch of computer science.
It focuses on creating machines that can perform tasks
that normally require human intellige

Chunk 2
nce.
AI is used in healthcare, education, banking, transportation
and entertainment.
Machine Learning is an important part of Artificial Intelligence.

Chunk 3

Deep Learning uses neural networks to learn complex patterns
from large amounts of data.


Metadata:
{'chunk_index': 0, 'source': 'ai_overview_v2.txt', 'char_count': 150}
{'char_count': 150, 'source': 'ai_overview_v2.txt', 'chunk_index': 1}
{'char_count': 90, 'source': 'ai_overview_v2.txt', 'chunk_index': 2}


In [16]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="practice3"
)


def add_document(collection, text, source_name, chunk_size):

    # Create chunks
    chunks = [
        text[i:i+chunk_size]
        for i in range(0, len(text), chunk_size)
    ]

    # Create IDs
    ids = [
        f"{source_name}_chunk_{i}"
        for i in range(len(chunks))
    ]

    # Create metadata
    metadatas = [
        {
            "source": source_name,
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(chunks)
    ]

    # Add to ChromaDB
    collection.add(
        documents=chunks,
        ids=ids,
        metadatas=metadatas
    )

    print(source_name, "added successfully!")
    print("Number of chunks:", len(chunks))


# Document 1
text1 = """
Artificial Intelligence allows computers to perform
tasks that normally require human intelligence.
AI is used in healthcare, education and transportation.
"""

add_document(
    collection,
    text1,
    "ai",
    100
)


# Document 2
text2 = """
Python is a programming language that is easy to learn.
Python is commonly used for Artificial Intelligence,
Machine Learning and Data Science.
"""

add_document(
    collection,
    text2,
    "python",
    100
)


# Document 3
text3 = """
Machine Learning allows computers to learn from data.
It is an important technology used in Artificial Intelligence.
"""

add_document(
    collection,
    text3,
    "machine_learning",
    100
)


# Total chunks
print("\nTotal chunks:", collection.count())


# Retrieval
question = "What is Machine Learning?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

print("\nQuestion:", question)

print("\nRetrieved results:")

for i, document in enumerate(results["documents"][0]):

    print("\nResult", i + 1)
    print(document)

    print("Metadata:")
    print(results["metadatas"][0][i])

ai added successfully!
Number of chunks: 2
python added successfully!
Number of chunks: 2
machine_learning added successfully!
Number of chunks: 2

Total chunks: 6

Question: What is Machine Learning?

Retrieved results:

Result 1

Machine Learning allows computers to learn from data.
It is an important technology used in Artific
Metadata:
{'chunk_index': 0, 'source': 'machine_learning', 'char_count': 100}

Result 2
lligence,
Machine Learning and Data Science.

Metadata:
{'char_count': 45, 'source': 'python', 'chunk_index': 1}

Result 3

Artificial Intelligence allows computers to perform
tasks that normally require human intelligence.
Metadata:
{'char_count': 100, 'chunk_index': 0, 'source': 'ai'}
